# Practice Session 06: PageRank

<font size="+2" color="blue">Additional results: spam/nonspam visualization</font>

# 1. Read host names

In [ ]:
import io
import gzip
import csv
import networkx as nx
import matplotlib.pyplot as plt

In [ ]:
INPUT_NODES_FILENAME = "webspam_uk2007-nodes.csv.gz"
INPUT_EDGES_FILENAME = "webspam_uk2007-edges.csv.gz"

In [ ]:
# Read the INPUT_NODES_FILENAME file into id2name, name2id, and id2label

id2name = {}
name2id = {}
id2label = {}

with gzip.open(INPUT_NODES_FILENAME, "rt", encoding="utf-8") as input_file:
    reader = csv.DictReader(input_file, delimiter=',', quotechar='"')
    for record in reader:
        id = int(record['nodeid'])
        name = record['hostname']
        label = record['label']
        
        id2name[id] = name
        name2id[name] = id
        id2label[id] = label

In [ ]:
# Leave as-is

print("Number of hosts: %s" % len(id2name))
print("%s: %s" % (id2name[107471], id2label[107471]))
print("%s: %s" % (id2name[3735], id2label[3735]))

In [ ]:
# Print the number and percentage of the hosts are spam, nonspam, and unlabeled. The latter should be the large majority. 
# Format the number with thousand separators and the percentage with one decimal 

from operator import countOf

spam_count = countOf(id2label.values(), 'spam')
nonspam_count = countOf(id2label.values(), 'nonspam')
unlabeled_count = countOf(id2label.values(), 'unlabeled')
count = len(id2label)

print(f'unlabeled: {unlabeled_count:,}\t({unlabeled_count/count:.1%})')
print(f'nonspam:   {nonspam_count:,}\t({nonspam_count/count:.1%})')
print(f'spam:      {spam_count:,}\t\t({spam_count/count:.1%})')

In [ ]:
# Load a subgraph of the input graph, as described above

g = nx.DiGraph()
spammywords = ['shop', 'directory', 'credit', 'mortgage', 'finance', 'debt', 'loan', 'discount', 'escort', 'xx', 'girl']
labels = ['spam', 'nonspam']

with gzip.open(INPUT_EDGES_FILENAME, "rt", encoding="utf-8") as input_file:
    reader = csv.DictReader(input_file, delimiter=',', quotechar='"')
    for record in reader:
        source = int(record['source'])
        destination = int(record['destination'])

        if not any((word in id2name[source] or word in id2name[destination]) for word in spammywords):
            continue
        if id2label[source] not in labels or id2label[destination] not in labels:
            continue
        # I use this structure so that in case it is not valid there is no need of doing further verifications and the code is cleaner

        g.add_node(id2name[source])
        g.add_node(id2name[destination])
        g.add_edge(id2name[source], id2name[destination])

In [ ]:
# Leave this code as-is, or modify slightly

colors = []
hostname_converted = {}

for hostname in g.nodes():
    # Assign colors to nodes according to spam/nonspam labels
    if id2label[name2id[hostname]] == 'spam':
        colors.append('red')
    elif id2label[name2id[hostname]] == 'nonspam':
        colors.append('lightgreen')
    else:
        colors.append('white')
    
    # Shorten the hostnames to generate labels    
    label = hostname.replace("www.", "").replace(".uk", "")
    hostname_converted[hostname] = label
    
# Notice that if you re-run this cell the layout will be different every time
plt.figure(figsize=(20, 20))
plt.axis('off')
pos = nx.spring_layout(g)
nx.draw_networkx(g, pos, with_labels=True, node_size=400, node_color=colors, labels=hostname_converted)

We can see in the plot above that most of the spam nodes have a high out-degree and therefore are more prone to be in the center of the graph layout, while the non-spam websites have a higher in-degree and normally those edges are towards other non-spam hosts. Similarly, spam pages seem to be the only ones that point to other spam hosts.

# 2. Compute the degree of each node

In [ ]:
# Leave this code as-is

id2degree = {}
N = len(id2name)
for nodeid in range(N):
    id2degree[nodeid] = 0

In [ ]:
with gzip.open(INPUT_EDGES_FILENAME, "rt", encoding="utf-8") as input_file:
    reader = csv.DictReader(input_file, delimiter=',', quotechar='"')
    for record in reader:
        source = int(record['source'])
        id2degree[source] += 1

In [ ]:
# Leave this cell as-is

for nodeid in [107471, 3735, 48842]:
    print("%s: degree %d" % (id2name[nodeid], id2degree[nodeid]))

# 3. Compute PageRank

In [ ]:
# Leave this cell as-is

ITERATIONS = 20
ALPHA = 0.92

pagerank = [1.0/N] * N
pagerank_aux = [0.0] * N

In [ ]:
# Compute PageRank
random_hop = (1.0-ALPHA)/N

for iteration in range(ITERATIONS):
    prev_iter = pagerank
    with gzip.open(INPUT_EDGES_FILENAME, "rt", encoding="utf-8") as input_file:
        reader = csv.DictReader(input_file, delimiter=',', quotechar='"')

        # Fill pagerank_aux for the current iteration
        for record in reader:
            source = int(record['source'])
            destination = int(record['destination'])
            pagerank_aux[destination] += pagerank[source]/id2degree[source]

        # Set pagerank of every node using ALPHA value and random hops
        pagerank = [ALPHA * pagerank_aux[i] + random_hop for i in range(N)]
        
        # Normalize pagerank
        aux_sum = sum(pagerank)
        pagerank = [pagerank[i]/aux_sum for i in range(N)]

        # Compute and print absolute difference
        delta = sum([abs(pagerank[i]-prev_iter[i]) for i in range(N)])
        print(f'Iteration {iteration} has absolute difference {delta:.4f}')

        # Reset pagerank_aux
        pagerank_aux = [0.0] * N

# 4. Nodes with largest values of PageRank

In [ ]:
def print_largest(ranks, id2name, id2label, count, reverse=False):
    hosts_by_score = sorted(enumerate(ranks), key=lambda x:x[1], reverse=reverse)
    print('┃{id:^9}┃{name:^34}┃{label:^13}┃{score:^10}┃'.format(id='HOSTID', name='NAME', label='LABEL', score='SCORE'))
    print('┃'+'━'*9+'╋'+'━'*34+'╋'+'━'*13+'╋'+'━'*10+'┃')
    for i in range(count):
        print('┃{id:^9}┃{name:^34}┃{label:^13}┃{score:^10.6f}┃'.format(id=hosts_by_score[i][0], name=id2name[hosts_by_score[i][0]], label=id2label[hosts_by_score[i][0]], score=hosts_by_score[i][1]))

In [ ]:
print_largest(pagerank, id2name, id2label, 20, reverse=True)

These sites are the top ones probably because they are very referenced by other hosts, but they do not reference many other, so they end up accumulating more score than others.

Most of the websites of the TOP-20 are government websites (".gov.uk" represent 65%), commercial websites (".co.uk" represent 25%) and organizations (".org.uk" are 10%). It makes kind of sense that government hosts make such a big percentage of the highest pagerank scores because probably government pages reference other government pages much more than non-government ones, and because of that, those hosts are very interconnected and therefore they accumulate more pagerank score.

# 5. Run non-spam PageRank

In [ ]:
# Compute id2nsdegree

id2nsdegree = {}
for nodeid in range(N):
    id2nsdegree[nodeid] = 0

with gzip.open(INPUT_EDGES_FILENAME, "rt", encoding="utf-8") as input_file:
    reader = csv.DictReader(input_file, delimiter=',', quotechar='"')
    for record in reader:
        source = int(record['source'])
        destination = int(record['destination'])
        if id2label[source] != 'spam' and id2label[destination] != 'spam':
            id2nsdegree[source] += 1

In [ ]:
# Leave this cell as-is

for nodeid in [107471, 1469, 48842]:
    print("%s: normal degree %d nospam degree %d" % (id2name[nodeid], id2degree[nodeid], id2nsdegree[nodeid]))

In [ ]:
# Compute non-spam PageRank

nspagerank = [1.0/N] * N
nspagerank_aux = [0.0] * N

random_hop = (1.0-ALPHA)/N

for iteration in range(ITERATIONS):
    prev_iter = nspagerank
    with gzip.open(INPUT_EDGES_FILENAME, "rt", encoding="utf-8") as input_file:
        reader = csv.DictReader(input_file, delimiter=',', quotechar='"')

        # Fill pagerank_aux for the current iteration
        for record in reader:
            source = int(record['source'])
            destination = int(record['destination'])
            if id2label[source] != 'spam' and id2label[destination] != 'spam':
                nspagerank_aux[destination] += nspagerank[source]/id2nsdegree[source]

        # Set pagerank of every node using ALPHA value and random hops
        nspagerank = [ALPHA * nspagerank_aux[i] + random_hop for i in range(N)]
        
        # Normalize pagerank
        aux_sum = sum(nspagerank)
        nspagerank = [nspagerank[i]/aux_sum for i in range(N)]

        # Compute and print absolute difference
        delta = sum([abs(nspagerank[i]-prev_iter[i]) for i in range(N)])
        print(f'Iteration {iteration} has absolute difference {delta:.4f}')

        # Reset pagerank_aux
        nspagerank_aux = [0.0] * N

In [ ]:
print_largest(nspagerank, id2name, id2label, 20, reverse=True)

The TOP-20 hosts are exactly the same, with a very slight change in the score of each website. Since there are less websites to compute the pagerank, all the scores are a little bit higer because the same overall score has to be divided among less nodes.

# 6. Compute spam gain

In [ ]:
# Print top 30 hosts by spam gain

pagerank_gain = [pagerank[i]/nspagerank[i] for i in range(N)]

hosts_by_score = sorted(enumerate(pagerank_gain), key=lambda x:x[1], reverse=True)
print('┃{name:^39}┃{label:^13}┃{gain:^10}┃{pr:^13}┃{nspr:^13}┃'.format(name='NAME', label='LABEL', gain='GAIN', pr='PAGERANK', nspr='NS-PAGERANK'))
print('┃'+'━'*39+'╋'+'━'*13+'╋'+'━'*10+'╋'+'━'*13+'╋'+'━'*13+'┃')
for i in range(50):
    print('┃{name:^39}┃{label:^13}┃{score:^10.2f}┃{pr:^13.1e}┃{nspr:^13.1e}┃'.format(name=id2name[hosts_by_score[i][0]], label=id2label[hosts_by_score[i][0]], score=hosts_by_score[i][1], pr=pagerank[hosts_by_score[i][0]], nspr=nspagerank[hosts_by_score[i][0]]))

Since in the non-spam pagerank, the spam websites had a score of 0 (approximately), and in the normal pagerank they had at most 1.4e-04, it makes sense that the growth of those spam accounts is much higher than the ones that were not spam, that is why the top ones are almost all spam websites.

For the two websites that are not spam: they were probably only pointed by spam websites, and that makes them lose an important amount of their score, therefore the gain is also high.

As I understand the idea behind the non-spam pagerank, it does not only involve pages that are marked as spam, but also pages that may be unlabeled, but are actually spam pages. That information can be used to label more websites as spam using gain.

# Extra section

In [ ]:
OUTPUT_PAGERANK = 'top_pagerank_hosts.csv'

with io.open(OUTPUT_PAGERANK, 'w') as output_file:
    writer = csv.writer(output_file, delimiter='\t', quotechar='"', lineterminator='\n')
    writer.writerow(['Hosts', 'Pagerank'])
    for i in range(N):
        if '.co.uk' in id2name[i]:
        	writer.writerow([i, pagerank[i]])

For creating the following graphs, first I unzipped the datasets, imported the edges (network), removed all unlabeled nodes (from 111k nodes to 5897 nodes), and then imported the data of pageranks. Then deleted all nodes that did not have a pagerank value. The following graph is the main component of the remaining network.

In [ ]:
from IPython.display import Image

In [ ]:
Image(url='style-pagerank.png')

In [ ]:
Image(url='style-degree.png')

We can see that the pages with high pagerank are very interconnected between them than with other spam pages, but they not have a significantly high degree. However, the spam pages seem to have much higher degree, because they point to many other websites but despite that they have not got any good pagerank score because legit pages do not point to them as much.

<font size="+2" color="#003300">I hereby declare that, except for the code provided by the course instructors, all of my code, report, and figures were produced by myself.</font>